In [1]:
import pandas as pd

df = pd.read_parquet('../data/Processed Positions Data.parquet')
df = df.drop(columns=['gameId', 'gameDateTimeEst_player', 'coachId'])
df.head()

,firstName,lastName,personId,playerteamName,gameType_player,win_player,home_player,numMinutes_player,points_player,assists_player,...,turnoversPercent,fgRelativePercent,3pRelativePercent,ftRelativePercent,3PA_rate,trueShooting,eFG,FTA_rate,USG,Position
0,Rudy,Gobert,203497.0,Timberwolves,Playoffs,0.0,0.0,29.316668,3.0,2.0,...,0.0400,-0.006,-0.429,0.174,0.000000,0.614754,0.5,0.500000,4.850405,C
1,Mike,Conley,201144.0,Timberwolves,Playoffs,0.0,0.0,16.933332,4.0,4.0,...,0.0400,-0.006,-0.429,-0.826,0.000000,0.500000,0.5,0.000000,12.205674,PG
2,Spencer,Jones,1642461.0,Nuggets,Playoffs,1.0,1.0,36.933334,20.0,1.0,...,0.0625,0.212,0.421,0.200,0.555556,1.012146,1.0,0.222222,12.854611,SF
3,Christian,Braun,1631128.0,Nuggets,Playoffs,1.0,1.0,30.900000,9.0,3.0,...,0.0625,0.034,0.121,0.200,0.400000,0.765306,0.7,0.400000,9.715799,SG
4,Jaylen,Clark,1641740.0,Timberwolves,Playoffs,0.0,0.0,11.333333,0.0,1.0,...,0.0000,-0.506,-0.429,-0.826,0.000000,0.000000,0.0,0.000000,3.647343,SG


In [2]:
print(len(df))
df = df.reset_index(drop=True)
len(df.columns)

873494


97

In [3]:
# Drop all team statistics
df = df.drop(columns=[c for c in df.columns if '_team' in c])

In [4]:
team_stats = ['benchPoints', 'pointsFastBreak', 'pointsFromTurnovers',
              'pointsInThePaint', 'pointsSecondChance', 'reboundsTeam',
              'turnoversTeam', 'pointDifferential']
df = df.drop(columns=team_stats)

In [5]:
len(df.columns)

67

In [6]:
numeric_df = df.select_dtypes(include='number')
numeric_df = numeric_df.drop(columns=['personId'])
cols = list(numeric_df.columns) + ['gameType_player']
X = df[cols]
y = df['Position']

In [7]:
X = pd.get_dummies(X, columns=['gameType_player'])

In [8]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X = pd.DataFrame(
    imputer.fit_transform(X),
    columns=X.columns,
    index = X.index
)

X = X.astype('float32')

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

# Feature Reduction Aimed at Lowering Multicollinearity

In [ ]:
from utils import prune_redundant_features

kept_features, drop_cols = prune_redundant_features(X)

,Feature 1,Feature 2,Pearson Correlation,Abs Correlation
1258,playerteamId_player,gameType_player_All-Star Game,1.000000,1.000000
2210,gameType_player_Playoffs,gameType_player_Regular Season,-0.981975,0.981975
476,fieldGoalsAttempted_player,fieldGoalsAttemptedPercent,0.979899,0.979899
231,points_player,pointsPercent,0.972415,0.972415
923,freeThrowsPercentage_player,ftRelativePercent,0.971273,0.971273
535,fieldGoalsMade_player,fieldGoalsMadePercent,0.966293,0.966293
1065,reboundsTotal_player,reboundsTotalPercent,0.965540,0.965540
600,fieldGoalsPercentage_player,fgRelativePercent,0.960524,0.960524
605,fieldGoalsPercentage_player,eFG,0.960083,0.960083
1863,pointsPercent,fieldGoalsMadePercent,0.957867,0.957867


In [ ]:
X = X.drop(columns=drop_cols)
print(len(X.columns))

30


In [13]:
X_sample = X.sample(n=100000, random_state=42)
y_sample = y.loc[X_sample.index]

In [15]:
%pip install boruta

  Using cached Boruta-0.4.3-py3-none-any.whl.metadata (8.8 kB)
Using cached Boruta-0.4.3-py3-none-any.whl (57 kB)
Note: you may need to restart the kernel to use updated packages.


# Boruta Selection

In [16]:
from sklearn.ensemble import RandomForestClassifier
from boruta import BorutaPy

rfc = RandomForestClassifier(n_estimators=200, n_jobs=-1, class_weight='balanced', random_state=42, max_depth=8)
boruta_selector = BorutaPy(rfc, n_estimators='auto', verbose=2, random_state=42, max_iter=100)
boruta_selector.fit(X_sample, y_sample)
important_features = X.columns[boruta_selector.support_]
weak_features = X.columns[boruta_selector.support_weak_]
print(important_features)
print(weak_features)

Iteration: 	1 / 100
Confirmed: 	0
Tentative: 	30
Rejected: 	0
Iteration: 	2 / 100
Confirmed: 	0
Tentative: 	30
Rejected: 	0
Iteration: 	3 / 100
Confirmed: 	0
Tentative: 	30
Rejected: 	0
Iteration: 	4 / 100
Confirmed: 	0
Tentative: 	30
Rejected: 	0
Iteration: 	5 / 100
Confirmed: 	0
Tentative: 	30
Rejected: 	0
Iteration: 	6 / 100
Confirmed: 	0
Tentative: 	30
Rejected: 	0
Iteration: 	7 / 100
Confirmed: 	0
Tentative: 	30
Rejected: 	0
Iteration: 	8 / 100
Confirmed: 	21
Tentative: 	1
Rejected: 	8
Iteration: 	9 / 100
Confirmed: 	21
Tentative: 	1
Rejected: 	8
Iteration: 	10 / 100
Confirmed: 	21
Tentative: 	1
Rejected: 	8
Iteration: 	11 / 100
Confirmed: 	21
Tentative: 	1
Rejected: 	8
Iteration: 	12 / 100
Confirmed: 	21
Tentative: 	1
Rejected: 	8
Iteration: 	13 / 100
Confirmed: 	21
Tentative: 	1
Rejected: 	8
Iteration: 	14 / 100
Confirmed: 	21
Tentative: 	1
Rejected: 	8
Iteration: 	15 / 100
Confirmed: 	21
Tentative: 	1
Rejected: 	8
Iteration: 	16 / 100
Confirmed: 	21
Tentative: 	1
Rejected: 	8
I